In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import numpy as np
import pywt
import soundfile as sf


WAVELET = "db12"
MODE = "periodization"   # IMPORTANT FIX


# ---------------------------------------------------------
# LOAD AUDIO
# ---------------------------------------------------------

def load_audio(path):
    data, sr = sf.read(path, dtype="float64")
    if data.ndim > 1:
        data = data[:, 0]
    return data, sr


# ---------------------------------------------------------
# SAVE AUDIO
# ---------------------------------------------------------

def save_audio(path, data, sr):
    sf.write(path, data, sr)


# ---------------------------------------------------------
# PAD FIX (CRITICAL)
# ---------------------------------------------------------

def pad_to_even(x):
    if len(x) % 2 != 0:
        x = np.append(x, 0)
    return x


# ---------------------------------------------------------
# DWT DECOMPOSITION (FIXED)
# ---------------------------------------------------------

def dwt_decompose(signal, levels):
    coeffs = signal.copy()
    details = []

    for _ in range(levels):
        coeffs = pad_to_even(coeffs)
        cA, cD = pywt.dwt(coeffs, WAVELET, mode=MODE)

        # FORCE SAME LENGTH (FIX)
        min_len = min(len(cA), len(cD))
        cA, cD = cA[:min_len], cD[:min_len]

        details.append(cD)
        coeffs = cA

    return coeffs, details


# ---------------------------------------------------------
# RECONSTRUCTION (FIXED)
# ---------------------------------------------------------

def dwt_reconstruct(cA, details):

    for cD in reversed(details):
        # force match length
        min_len = min(len(cA), len(cD))
        cA = cA[:min_len]
        cD = cD[:min_len]

        cA = pywt.idwt(cA, cD, WAVELET, mode=MODE)

    return cA


# ---------------------------------------------------------
# LEVEL CALC
# ---------------------------------------------------------

def get_levels(n, msg_len):
    return max(1, int(np.floor(np.log2(n / msg_len))))


# ---------------------------------------------------------
# TEXT ↔ BITS
# ---------------------------------------------------------

def text_to_bits(text):
    return np.array(
        [int(b) for c in text for b in format(ord(c), "08b")],
        dtype=np.float64
    )


def bits_to_text(bits):
    bits = bits.astype(int)
    out = []

    for i in range(0, len(bits), 8):
        byte = bits[i:i+8]
        out.append(chr(int("".join(byte.astype(str)), 2)))

    return "".join(out)


# ---------------------------------------------------------
# SCALE
# ---------------------------------------------------------

def scale_factor(x):
    return np.max(np.abs(x)) + 1e-6


# ---------------------------------------------------------
# EMBED
# ---------------------------------------------------------

def embed_message(audio, text):

    bits = text_to_bits(text)
    n = len(bits)

    levels = get_levels(len(audio), n)

    cA, details = dwt_decompose(audio, levels)

    band = details[-1]

    if len(band) < n:
        raise ValueError("Audio too small")

    scale = scale_factor(band)

    band[:n] = bits / scale
    details[-1] = band

    stego = dwt_reconstruct(cA, details)

    return stego, scale, levels, n


# ---------------------------------------------------------
# EXTRACT
# ---------------------------------------------------------

def extract_message(stego, scale, levels, n):

    cA, details = dwt_decompose(stego, levels)

    band = details[-1]

    bits = (band[:n] * scale > 0.5).astype(int)

    return bits_to_text(bits)


# ---------------------------------------------------------
# MAIN
# ---------------------------------------------------------

audio_path = input("Enter WAV filename: ")
message = input("Enter secret message: ")
output = "stego.wav"

audio, sr = load_audio(audio_path)

stego, scale, levels, n = embed_message(audio, message)

recovered = extract_message(stego, scale, levels, n)

save_audio(output, stego, sr)

print("\nMessage embedded successfully!")
print("Recovered:", recovered)
print("Output file:", output)

In [ ]:
files.download("")

**Observations**

- Hidden message was extracted correctly.
- Average execution time: 2–4 seconds.
- Distortion was minimal across most audio samples.
- DWT showed better imperceptibility than LSB.